# BIL458 Computer Vision — Term Project

**Title:** CancerAI: A Comparative Study of Deep Learning Strategies for Multi-Modal Medical Image Classification

**Author:** Farah Alhasan (21120205709)

**Instructor:** Assoc. Prof. Murat Gezer

> Before submission: *Kernel → Restart & Run All* and confirm the notebook runs without errors.
>
> The 48 runs are already saved under `results/`. `RUN_FULL_STUDY` stays `False`, so a fresh run loads those scores, the figures, and the unit tests instead of training again. The written report is `deliverables/BIL458_CancerAI_Report.pdf`. The ten-slide talk is `deliverables/BIL458_CancerAI_Slides.pptx`. The public page is https://43a3d99492816eb041.gradio.live.

This notebook is the experiment record for four separate classification tasks: brain MRI, breast histopathology, skin lesions, and lung CT. The labels are tumour and lesion categories. The project is not a universal cancer detector and not a clinical device.


## 0. Environment and reproducibility

Main question: how do CNN training strategies and data augmentation change macro-F1 across these four modalities?

- H1. Fine-tuned ResNet18 outperforms a custom CNN and a frozen ResNet18.
- H2. Training augmentation improves generalisation, especially on imbalanced data.
- H3. The gain differs by modality.
- H4. Accuracy hides minority-class errors. Macro-F1 and per-class recall expose them.

Seeds for training are 42, 123, and 2026. The split seed is 42 and is not changed by the training seed. Reported scores are the mean and the sample standard deviation across the three runs.

On Google Colab the notebook file is uploaded by itself, so the first code cell clones [CancerAI](https://github.com/FarahBebs/CancerAI) and installs the libraries. The image collections are not in that repository. Cells that need the files say so and continue. The saved scores and figures still load.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def find_project(start):
    for candidate in (start, start.parent, Path("/content/CancerAI")):
        if (candidate / "cancerai" / "__init__.py").exists():
            return candidate.resolve()
    return None

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

ROOT = find_project(Path.cwd().resolve())
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/CancerAI")
    if not (ROOT / "cancerai" / "__init__.py").exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/FarahBebs/CancerAI.git",
            str(ROOT),
        ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "numpy", "pillow", "scikit-learn", "medmnist", "tqdm",
    ])
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "pillow-heif"])

if ROOT is None or not (ROOT / "cancerai" / "__init__.py").exists():
    raise FileNotFoundError(
        "The cancerai package is not here. On Colab, run this cell with internet access. "
        "On your computer, open the notebook from the CancerAI project folder."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project", ROOT)
print("colab", IN_COLAB)


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import platform
import torch
from cancerai.training import get_device, seed_everything

SEED = 42
seed_everything(SEED)
print("python", platform.python_version())
print("torch", torch.__version__)
print("device", get_device())
print("project", ROOT)


## 1. Datasets

The train / validation / test membership is written once, to `results/splits/`, and later runs reload it. That is the leakage control this notebook can actually enforce.

- Breast: patient id from the BreakHis folder name. The code rejects a patient that appears in two splits.
- Skin: official DermaMNIST split. Lesion ids are not in the file, so lesion-level independence is not claimed.
- Brain MRI and lung CT: filenames were checked. They do not yield a patient or scan id. Those results are not described as patient-level.


In [ ]:
from cancerai.datasets import audit_datasets, describe, prepare

try:
    report = audit_datasets()
except (FileNotFoundError, RuntimeError) as exc:
    report = None
    print(exc)
    print("The image collections are not in this runtime. The saved scores below do not need them.")

if report is not None:
    print("data root:", report["data_root"])
    for name, entry in report["datasets"].items():
        print("\n==", name, "==")
        print("patient_independent:", entry["patient_independent"])
        print(entry["leakage_note"])
        for split, counts in entry["counts"].items():
            print(f"  {split}: {sum(counts.values())}")


In [ ]:
# One unaugmented bundle per modality. The same split file feeds every model.
DATA_READY = True
for name in ("brain", "breast", "skin", "lung"):
    try:
        bundle = prepare(name, augment=False)
    except (FileNotFoundError, RuntimeError, OSError) as exc:
        DATA_READY = False
        print(name, "skipped:", exc)
        continue
    print(describe(bundle))
    print()


## 2. Preprocessing and augmentation

Every model sees a 224×224 RGB image normalized with the ImageNet mean and standard deviation, so the comparison is not confounded by a different input pipeline.

Training augmentation depends on the modality:

- Breast: flips, ±20° rotation, mild brightness and contrast. No hue shift, because stain colour carries information.
- Skin: horizontal flip, ±15° rotation, mild brightness, contrast, and saturation. No hue shift.
- Brain MRI: ±10° rotation. No left-right flip.
- Lung CT: ±8° rotation. No flip and no colour change.

Validation and test images are only resized and normalized.


In [ ]:
import matplotlib.pyplot as plt
from cancerai.datasets import build_transform, prepare

def names(dataset, train, augment):
    transform = build_transform(dataset, train=train, augment=augment)
    return [type(step).__name__ for step in transform.transforms]

for dataset in ("brain", "breast", "skin", "lung"):
    print(dataset, "train", names(dataset, True, True))
print("eval", names("brain", False, False))

try:
    bundle = prepare("brain", augment=False)
    fig, axes = plt.subplots(1, 4, figsize=(10, 2.6))
    for axis, index in zip(axes, range(4)):
        axis.imshow(bundle.train_ds.raw(index))
        axis.set_title(bundle.classes[int(bundle.train_ds.rows[index]["label"])], fontsize=8)
        axis.axis("off")
    fig.suptitle("Brain MRI training examples (no augmentation)")
    plt.show()
except (FileNotFoundError, RuntimeError, OSError) as exc:
    print("Example images skipped:", exc)


## 3. Baseline

The baseline is a four-block CNN trained from scratch: convolution, ReLU, max pooling, then dropout and a linear layer. There is no ImageNet initialization.

A convolution reads a local patch,

$$Y(i,j)=\sum_{u=0}^{k-1}\sum_{v=0}^{k-1} X(i+u,j+v)\,K(u,v)$$

and ReLU is $f(x)=\max(0,x)$. One convolutional layer costs about $O(HW C_{in} C_{out} K^2)$ operations. The custom network uses four convolutions. ResNet18 repeats that pattern through a much deeper residual backbone, which is why it has more parameters and a higher cost. Freezing the backbone trains only the new head.


## 4. Proposed method

The loss is weighted cross-entropy. For a one-hot label this is

$$L=-\sum_{c=1}^{C} w_c y_c \log \hat y_c$$

with $w_c = N / (C n_c)$, rescaled so the weights average to 1. Softmax is inside the loss. The network returns logits.

Macro-F1 gives every class the same weight:

$$F1_{macro}=\frac{1}{C}\sum_{c=1}^{C}\frac{2 P_c R_c}{P_c+R_c}$$

The three ResNet18 strategies are: frozen backbone, fine-tuning with augmentation, and fine-tuning without augmentation. ROC-AUC, when it is defined, is one-versus-rest with macro averaging.


In [ ]:
from cancerai.models import build_model, count_params

for name, freeze, pretrained in (
    ("custom_cnn", False, False),
    ("resnet18", True, False),
    ("resnet18", False, False),
):
    model = build_model(name, n_classes=4, freeze_backbone=freeze, pretrained=pretrained)
    total, trainable = count_params(model)
    label = "frozen head only" if freeze else name
    print(f"{label:22}  total {total:10,d}  trainable {trainable:10,d}")


## 5. Training

Shared loop in `cancerai.training.train_model`: AdamW, weight decay 1e-4, cosine decay to 1e-6, batch size 32. The checkpoint is the epoch with the best validation macro-F1. Patience is 3 and the minimum improvement is 0.001. The test set is scored only after that checkpoint is restored.

| Configuration | Epoch cap | Learning rate |
|---|---:|---:|
| Custom CNN | 10 | 1e-3 |
| Frozen ResNet18 | 8 | 1e-3 |
| Fine-tuned ResNet18, with augmentation | 8 | 1e-4 |
| Fine-tuned ResNet18, without augmentation | 8 | 1e-4 |


In [ ]:
RUN_FULL_STUDY = False  # The 48 runs are already saved. True trains any run that has no metrics file.

if RUN_FULL_STUDY:
    from experiments.run_experiments import main
    main([])
else:
    print("Using the saved 48 runs in results/metrics.")
    print("Set RUN_FULL_STUDY = True only to retrain. A finished metrics file is skipped unless --force is passed to the script.")


## 6. Evaluation (at least 3 seeds, mean ± std)

Primary metric: test macro-F1. Also accuracy, macro precision, macro recall, the confusion matrix, one-vs-rest macro ROC-AUC when every required class is present, training time, and single-image inference time.


In [ ]:
from experiments.summarize import comparison_tables, load_runs

runs = load_runs()
print(comparison_tables(runs))
print("metric files found:", len(runs))


In [ ]:
from IPython.display import Image, Markdown, display

comparison = ROOT / "results" / "plots" / "macro_f1_comparison.png"
display(Markdown("**Test macro-F1 by modality and training strategy. The table above is the mean and the sample standard deviation.**"))
display(Image(filename=str(comparison)))

## 7. Ablation

The ablation is `finetune_aug` against `finetune_no_aug`. Both fine-tune ResNet18 with the same learning rate, the same split, and the same seeds. Only the training transform changes. Validation and test stay unaugmented, so a higher training accuracy from augmentation is not counted as a better model unless the test macro-F1 moves.


## 8. Failure analysis

Each run writes its highest-confidence test mistakes to `results/plots/` and a Grad-CAM figure to `results/gradcam/`. The fine-tuned model with augmentation shows three failures that the average hides.

- **BreakHis.** Phyllodes tumour F1 is 0.007. About 143 of 235 test slides are called fibroadenoma. The two benign fibroepithelial patterns are close. Phyllodes has the highest class weight, about 4.1, because the training split has only 60 slides, and the model still misses the class. Grad-CAM on those mistakes sits on the tissue, so this is a look-alike class rather than a border artefact. Ductal carcinoma is also confused with lobular carcinoma.
- **DermaMNIST.** Melanoma F1 is 0.568 with precision 0.484. About 140 nevi are called melanoma. Nevi are 1,341 of 2,005 test images, so accuracy (0.787) looks kinder than macro-F1 (0.719).
- **Brain MRI and lung CT.** The scores are high, and the filenames do not identify the patient. A near-perfect brain score is not evidence that two slices of one scan were kept apart. Lung benign F1 rests on 18 test slices.

The Gradio page is for the presentation. It is not evidence that a prediction is correct. Grad-CAM is not a tumour outline.


In [ ]:
import numpy as np
from cancerai.metrics import worst_mistakes

# Shape of the failure picker, on a toy prediction vector.
y_true = np.array([0, 1, 1, 0])
y_pred = np.array([0, 0, 1, 1])
probs = np.array([[0.9, 0.1], [0.8, 0.2], [0.4, 0.6], [0.3, 0.7]])
print("highest-confidence mistakes:", worst_mistakes(y_true, y_pred, probs, k=5))


In [ ]:
display(Markdown("**BreakHis confusion matrix, fine-tuned ResNet18 with augmentation, seed 42.**"))
display(Image(filename=str(ROOT / "results" / "plots" / "breast__finetune_aug__seed42_confusion.png")))
display(Markdown("**Grad-CAM on high-confidence BreakHis mistakes. The heat map is not a tumour outline.**"))
display(Image(filename=str(ROOT / "results" / "gradcam" / "breast__finetune_aug__seed42.png")))
display(Markdown("**DermaMNIST confusion matrix, fine-tuned ResNet18 with augmentation, seed 42.**"))
display(Image(filename=str(ROOT / "results" / "plots" / "skin__finetune_aug__seed42_confusion.png")))

## 9. Unit tests

These tests check the loss formula, the patient split, the augmentation policy, the macro-F1 definition, and the parameter counts. They do not need the image files.


In [ ]:
import importlib.util
import sys
import unittest
from pathlib import Path

def project_root():
    here = Path.cwd().resolve()
    for candidate in (here, here.parent, Path("/content/CancerAI")):
        if (candidate / "tests" / "test_core.py").exists():
            return candidate.resolve()
    return None

root = project_root()
if root is None:
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise FileNotFoundError(
            "tests/test_core.py is not in this folder. Open the notebook from the CancerAI project."
        ) from exc
    import subprocess
    root = Path("/content/CancerAI")
    if not (root / "tests" / "test_core.py").exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/FarahBebs/CancerAI.git",
            str(root),
        ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pillow", "scikit-learn",
    ])

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

spec = importlib.util.spec_from_file_location("cancerai_test_core", root / "tests" / "test_core.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
suite = unittest.defaultTestLoader.loadTestsFromModule(module)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Unit tests failed"
print("Tests passed")


## 10. Demo

The public page and the local page are the same app. A new visit opens on the breast histopathology example, with the category, the broader group, the full score list, and the Grad-CAM already filled in.

https://43a3d99492816eb041.gradio.live

The same page is on this machine at http://127.0.0.1:7860. The public link is a temporary Gradio tunnel. It lasts up to about a week, and it stops if this computer sleeps or the process exits.

![The CancerAI page, local and public, opened on the breast example](figures/site-full.png)

The broader group is the category probabilities added together with a fixed map. Brain groups are tumor or no tumor, and tumor is not reported as cancer. Breast groups are benign or malignant. Skin groups are cancer-related or not, with actinic keratoses kept in the cancer-related group because that is the dataset convention. Lung groups are malignant or not.

The demo checkpoint is the fine-tuned ResNet18 with augmentation whose validation macro-F1 was best among the three seeds. Those weights are `models/brain.pt`, `models/breast.pt`, `models/skin.pt`, and `models/lung.pt`.


## Conclusion

Fine-tuning ResNet18 leads on every modality. Test macro-F1 is 0.989 ± 0.004 on brain MRI, 0.506 ± 0.031 on breast, 0.719 ± 0.014 on skin, and 0.946 ± 0.021 on lung CT. A frozen backbone is useful on brain MRI and lung CT and is not useful on skin, where it scores below the custom CNN. Augmentation raises the patient-level breast score and lowers the skin and lung scores.

H1 and H3 hold. H2 holds for breast only. H4 holds for breast and skin: accuracy hides the missed phyllodes class and the over-called melanoma class.

Only BreakHis supports a patient-level claim, and on that set the headline score is about 0.51. Brain MRI and lung CT stay in the study with that limit written down. A next experiment should replace those two collections with a release that carries a scan id. Until then the page that shows a prediction is a teaching demo.
